In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


In [ ]:
DATA_PATH = Path("../data/raw/seg_train")

print(DATA_PATH.exists())


In [ ]:
class_names = []

for folder in DATA_PATH.iterdir():
    if folder.is_dir():
        class_names.append(folder.name)

class_names.sort()

print(class_names)
print("Number of classes:", len(class_names))


In [ ]:
image_extensions = [".jpg", ".jpeg", ".png"]

class_counts = {}

for class_name in class_names:
    class_path = DATA_PATH / class_name
    count = 0

    for file in class_path.iterdir():
        if file.suffix.lower() in image_extensions:
            count += 1

    class_counts[class_name] = count

print(class_counts)


In [ ]:
distribution_df = pd.DataFrame(
    class_counts.items(),
    columns=["class", "image_count"]
)

print(distribution_df)


In [ ]:
distribution_df.plot(
    x="class",
    y="image_count",
    kind="bar",
    legend=False
)

plt.title("Images per Class")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=45)
plt.show()


In [ ]:
image_data = []

for class_name in class_names:
    class_path = DATA_PATH / class_name

    for file_path in class_path.iterdir():
        if file_path.suffix.lower() in image_extensions:
            try:
                image = Image.open(file_path)
                width, height = image.size

                image_data.append({
                    "class": class_name,
                    "filename": file_path.name,
                    "width": width,
                    "height": height,
                    "format": image.format,
                    "mode": image.mode
                })

                image.close()

            except:
                print("Problem with:", file_path)


In [ ]:
images_df = pd.DataFrame(image_data)

print(images_df.head())
print("Total images:", len(images_df))
print(images_df[["width", "height"]].describe())


In [ ]:
images_df["resolution"] = (
    images_df["width"].astype(str)
    + "x"
    + images_df["height"].astype(str)
)

print(images_df["resolution"].value_counts().head(10))


In [ ]:
print(images_df["format"].value_counts())
print(images_df["mode"].value_counts())


In [ ]:
small_images = images_df[
    (images_df["width"] < 100)
    |
    (images_df["height"] < 100)
]

print("Small images:", len(small_images))


In [ ]:
broken_images = []

for class_name in class_names:
    class_path = DATA_PATH / class_name

    for file_path in class_path.iterdir():
        if file_path.suffix.lower() in image_extensions:
            try:
                image = Image.open(file_path)
                image.verify()
                image.close()

            except:
                broken_images.append(file_path)

print("Broken images:", len(broken_images))


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i, class_name in enumerate(class_names):
    class_path = DATA_PATH / class_name
    images = list(class_path.glob("*.jpg"))
    image = Image.open(images[0])

    axes[i].imshow(image)
    axes[i].set_title(class_name)
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flatten()

for i in range(len(small_images)):
    row = small_images.iloc[i]

    image_path = DATA_PATH / row["class"] / row["filename"]
    image = Image.open(image_path)

    axes[i].imshow(image)
    axes[i].set_title(
        f'{row["class"]} - {row["width"]}x{row["height"]}'
    )
    axes[i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
OUTPUT_PATH = Path("../data/processed")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

images_df.to_csv(
    OUTPUT_PATH / "image_metadata.csv",
    index=False
)

print("Metadata saved")
